# ORBIX — New observed-data inference

This notebook creates an explicitly named experiment using the SB1/SB2 RV
pipeline. Archived runs are research evidence and are not overwritten.
Run completion, MCMC diagnostics and residual adequacy are separate criteria.
The four schemes answer different conditional questions; fixed parameters
are not exact measurements. Review `docs/limitations.rst` before interpretation.

Open this notebook with `corrida_masiva` as the working directory.

In [1]:
from pathlib import Path
import sys, json, shutil, hashlib, re
from dataclasses import asdict
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'orbix/benchmark.py').exists())
BASE = REPO/'análisis_desempeño/corridas_masivas'
if not (REPO / 'orbix').is_dir():
    raise RuntimeError('Use corrida_masiva as the working directory.')
sys.path.insert(0, str(REPO))
from orbix import config, ajuste, masiva
SYSTEMS = config.cargar_sistemas(BASE / 'config/sistemas.csv')
PRIORS = config.cargar_priors(BASE / 'config/priors.csv')
display(SYSTEMS[['sistema','tipo','esquemas','P_lit','e_lit','epoch_format_in','epoch_format_work']])
display(PRIORS)

,sistema,tipo,esquemas,P_lit,e_lit,epoch_format_in,epoch_format_work
0,HIP109951,SB1,"free,fix_P,fix_e,fix_eP",54.7960,0.40700,year_besselian,year_besselian
1,HIP89000,SB1,"free,fix_P,fix_e,fix_eP",0.5460,0.30300,year_besselian,year_besselian
2,HIP89000,SB2,"free,fix_P,fix_e,fix_eP",0.5460,0.30100,year_besselian,year_besselian
3,HIP117186,SB2,"free,fix_P,fix_e,fix_eP",0.2350,0.32300,year_besselian,year_besselian
4,HIP20601,SB2,"free,fix_P,fix_e,fix_eP",0.4280,0.85200,year_besselian,year_besselian
5,HIP14157,SB2,"free,fix_P,fix_e,fix_eP",0.1190,0.76100,year_besselian,year_besselian
6,2MASS_J05215658+4359220,SB1,"free,fix_P,fix_e,fix_eP",83.2050,0.00476,TJD,TJD
7,HD170582,SB1,"free,fix_P,fix_e,fix_eP",16.8722,0.01330,TJD,TJD
8,HD50526,SB1,"free,fix_P,fix_e,fix_eP",6.7048,0.04974,HJD,RJD


,sistema,tipo,parametro,familia,p1,p2,unidad,fuente,nota
0,*,*,P,flat,,,,default ORBIX,uniforme en ventana LS amplia derivada de los ...
1,*,*,e,flat,,,,Videla et al. 2022,"uniforme en rango fisico [0, 0.999)"
2,*,*,T0,flat,,,,Videla et al. 2022,"uniforme en fase [0, 1); reemplaza beta(2,2)"
3,*,*,V0,flat,,,,default ORBIX,uniforme en ventana amplia desde la escala de RV
4,*,SB1,K,flat,,,,default ORBIX,uniforme positiva con Kmax amplio desde la esc...
5,*,SB2,K1,flat,,,,default ORBIX,uniforme positiva con Kmax amplio desde v1
6,*,SB2,K2,flat,,,,default ORBIX,uniforme positiva con Kmax amplio desde v2


## Prespecified configuration

Choose a new experiment name. The saved historical configurations use rounded
fixed HIP periods; this can be scientifically inappropriate for precise long-baseline
data. Choosing a different fixed value defines a different experiment and must
be documented before fitting. Defaults below reproduce the numerical budget,
not a guarantee of posterior convergence or a global optimum.

In [2]:
EXPERIMENT = 'new_observed_experiment'
SELECT_SYSTEMS = None
SELECT_SCHEMES = None
RUN_INFERENCE = False

if EXPERIMENT.lower() in {'tablas','draws','fase1','fase1_revision_20261027','observados_final_20261029'} or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,63}',EXPERIMENT):
    raise ValueError('Choose a new, simple experiment directory name.')
OUTPUT = BASE / 'resultados' / EXPERIMENT
SETTINGS = ajuste.Ajustes(extra={'timeout_opt':60,'timeout_sample':1800})
PATHS = ajuste.Rutas(BASE, REPO / 'modelos', OUTPUT)
print('Output:', OUTPUT)
print(asdict(SETTINGS))

Output: C:\Users\claud\OneDrive\Desktop\info\Tesis\ORBIX\análisis_desempeño\corridas_masivas\resultados\new_observed_experiment
{'n_inicios_SB1': 200, 'n_inicios_SB2': 300, 'algoritmo': 'lbfgs', 'iter_opt': 4000, 'tol_obj': 1e-06, 'tol_grad': 1e-06, 'tol_param': 1e-06, 'history_size': 20, 'cadenas': 4, 'iter_warmup': 1000, 'iter_sampling': 1000, 'adapt_delta': 0.95, 'max_treedepth': 15, 'semilla': 20261005, 'umbral_rhat': 1.01, 'hdi_prob': 0.95, 'siembra_P': 'global', 'k_picos': 8, 'frac_picos': 0.6, 'refinar_picos': True, 'n_frecuencias_refino': 41, 'sig_figs': 18, 'umbral_chi2': 5.0, 'extra': {'timeout_opt': 60, 'timeout_sample': 1800}}


## Execute only after reviewing the protocol

Set `RUN_INFERENCE=True` above to opt in. Successful computations are resumed;
diagnostic warnings are preserved. The protocol snapshot prevents resuming the
same named experiment with silently changed configuration or source code.

In [3]:
if RUN_INFERENCE:
    sources = list((REPO/'orbix').glob('*.py')) + list((REPO/'modelos').glob('sb*_K*.stan'))
    sources += [REPO/'modelos/kepler_eq.hpp', BASE/'config/sistemas.csv', BASE/'config/priors.csv']
    protocol = {'settings':asdict(SETTINGS), 'systems':SELECT_SYSTEMS, 'schemes':SELECT_SCHEMES,
                'sha256':{str(p.relative_to(BASE.parent)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(sources)}}
    OUTPUT.mkdir(parents=True, exist_ok=True)
    saved = OUTPUT / 'protocol.json'
    if saved.exists() and json.loads(saved.read_text(encoding='utf-8')) != protocol:
        raise RuntimeError('Protocol differs: choose a new experiment name.')
    saved.write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    snapshot = OUTPUT/'config'; snapshot.mkdir(exist_ok=True)
    for name in ['sistemas.csv','priors.csv']:
        shutil.copy2(BASE/'config'/name,snapshot/name)
    tables = masiva.correr_todo(SYSTEMS,PRIORS,PATHS,SETTINGS,
        esquemas=SELECT_SCHEMES,solo=SELECT_SYSTEMS,reanudar=True)
    display(tables['corridas'])
else:
    print('Inference disabled. No optimizer, sampler or model compilation was executed.')

Inference disabled. No optimizer, sampler or model compilation was executed.
